# 02 — BaseAgent Contract (`core/base_agent.py`)
Three classes define the agent contract:
- **`AgentRequest`** — what any agent receives (query, intent, filters, context, etc.)
- **`AgentResult`** — what any agent returns (success, data, summary, sources, confidence)
- **`BaseAgent`** — abstract class; subclasses implement `_execute()`, never `execute()`

Key design: `execute()` catches ALL exceptions and returns `AgentResult(success=False)` — agents never crash the graph.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'codebase', 'codebase', 'src'))
os.makedirs("logs", exist_ok=True)

## 1. AgentRequest — Build a Request

In [ ]:
from core.base_agent import AgentRequest

req = AgentRequest(
    query="What is the GRR for retention last month?",
    intent="metric_analysis",
    data_products=["retention"],
    time_range="last_month",
)
print("query_id    :", req.query_id)   # auto-generated
print("query       :", req.query)
print("intent      :", req.intent)
print("data_products:", req.data_products)
print("time_range  :", req.time_range)
print("context     :", req.context)   # empty dict by default
print("filters     :", req.filters)

## 2. AgentResult — Build a Result

In [ ]:
from core.base_agent import AgentResult

result = AgentResult(
    agent_name="my_agent",
    success=True,
    data={"grr": 87.5, "nrr": 105.2},
    summary="GRR is 87.5% — above the 85% threshold.",
    sources=["analytics.retention_metrics"],
    confidence=0.95,
)
print("agent_name      :", result.agent_name)
print("success         :", result.success)
print("data            :", result.data)
print("summary         :", result.summary)
print("sources         :", result.sources)
print("confidence      :", result.confidence)
print("execution_time_ms:", result.execution_time_ms)
print("timestamp       :", result.timestamp)

## 3. AgentResult.to_dict() — Serialization for API responses

In [ ]:
import json
d = result.to_dict()
print(json.dumps(d, indent=2))

## 4. Implement a Custom Agent

In [ ]:
from core.base_agent import BaseAgent, AgentRequest, AgentResult

class EchoAgent(BaseAgent):
    name = "echo_agent"
    description = "Returns the query back as its result"
    capabilities = ["echo"]

    def _execute(self, request: AgentRequest) -> AgentResult:
        return AgentResult(
            agent_name=self.name,
            success=True,
            data={"echoed_query": request.query},
            summary=f"Echo: {request.query}",
            confidence=1.0,
        )

agent = EchoAgent()
req = AgentRequest(query="Hello, Copilot!")
result = agent.execute(req)  # calls execute() not _execute()

print("success  :", result.success)
print("summary  :", result.summary)
print("exec_ms  :", result.execution_time_ms)  # timed by BaseAgent

## 5. Error Handling — Agent never raises

In [ ]:
class BrokenAgent(BaseAgent):
    name = "broken_agent"
    capabilities = []

    def _execute(self, request):
        raise ValueError("Simulated data source failure!")

broken = BrokenAgent()
result = broken.execute(AgentRequest(query="test"))
print("success :", result.success)     # always returns, never raises
print("error   :", result.error)
print("summary :", result.summary)

## 6. health_check()

In [ ]:
class HealthyAgent(BaseAgent):
    name = "healthy_agent"
    capabilities = ["read", "write"]
    def _execute(self, request): pass

h = HealthyAgent()
print(h.health_check())

## 7. AgentRequest with Context (for write operations)

In [ ]:
req = AgentRequest(
    query="create ticket for retention anomaly",
    intent="write_ticket",
    context={
        "ticket_summary": "GRR dropped below 85% threshold",
        "priority": "High",
        "labels": ["auto-dq-alert", "retention"],
    },
    data_products=["retention"],
)
print("context:", req.context)
print("query_id:", req.query_id)  # unique per request